In [ ]:
import gc
import json
import os
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import whisperx
from whisperx.audio import N_SAMPLES, log_mel_spectrogram

SOURCE_ROOT = Path('..')

device = 'cuda'
compute_type = 'float16'

model_big_dir = 'whisper_large-v3'
model_turbo_dir = 'whisper_large-v3-turbo'
model_a_dir = 'whisperx-align_ro'

TRANSCRIBE_BATCH_SIZE = 32
FILTER_INFER_BATCH_SIZE = 96
ALIGN_INFER_BATCH_SIZE = 96


def cuda_cleanup() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()


def batched_ranges(total: int, batch_size: int):
    for i in range(0, total, batch_size):
        yield i, min(i + batch_size, total)

In [ ]:
def chunk_audio(
    wav: torch.Tensor,
    sr: int = 16000,
    silence_ms: int = 1000,
) -> List[Tuple[int, int]]:
    wav_tensor = torch.from_numpy(wav).to(device)
    speech_ts = get_speech_timestamps(
        wav_tensor,
        vad_model,
        sampling_rate=sr,
        min_silence_duration_ms=silence_ms,
        speech_pad_ms=0,
    )

    chunks = [(s['start'], s['end']) for s in speech_ts]

    return chunks


def detect_languages_batched(
    wav: np.ndarray,
    chunks: List[Tuple[int, int]],
    infer_batch_size: int,
) -> List[Tuple[str, float]]:
    detected: List[Tuple[str, float]] = []
    model_n_mels = turbo_model.model.feat_kwargs.get('feature_size')

    for lo, hi in batched_ranges(len(chunks), infer_batch_size):
        batch = chunks[lo:hi]
        mel_batch = []

        for start, end in batch:
            audio_chunk = wav[start:end]
            mel = log_mel_spectrogram(
                audio_chunk[:N_SAMPLES],
                n_mels=model_n_mels if model_n_mels is not None else 80,
                padding=0
                if audio_chunk.shape[0] >= N_SAMPLES
                else N_SAMPLES - audio_chunk.shape[0],
            )
            mel_batch.append(mel)

        if not mel_batch:
            continue

        with torch.no_grad():
            mel_tensor = torch.stack(mel_batch).to(device)
            encoder_output = turbo_model.model.encode(mel_tensor)
            results = turbo_model.model.model.detect_language(encoder_output.cpu())

        for result in results:
            lang_token, lang_prob = result[0]
            detected.append((lang_token[2:-2], float(lang_prob)))

    return detected


def filter_non_ro(path: str, infer_batch_size: int = FILTER_INFER_BATCH_SIZE) -> None:
    sr = 16000

    wav = whisperx.load_audio(path, sr=sr)
    chunks = chunk_audio(wav)
    detected_langs = detect_languages_batched(
        wav, chunks, infer_batch_size=infer_batch_size
    )

    mask = np.ones(wav.shape[0], dtype=bool)

    for (start, end), (lang, _) in zip(chunks, detected_langs):
        if lang != 'ro':
            mask[start:end] = False

    mask_path = os.path.splitext(path)[0] + '.mask'
    mask = np.packbits(mask)
    mask.tofile(mask_path)


def transcribe(path: str, batch_size: int = TRANSCRIBE_BATCH_SIZE) -> None:
    sr = 16000
    audio = whisperx.load_audio(path, sr=sr)

    mask_path = os.path.splitext(path)[0] + '.mask'
    mask = np.fromfile(mask_path, dtype=np.uint8)
    mask = np.unpackbits(mask)[: audio.shape[0]].astype(bool)
    audio[~mask] = 0.0

    result = model_big.transcribe(audio, batch_size=batch_size, language='ro')
    out_path = os.path.splitext(path)[0] + '.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

In [ ]:
# Phase 1: load only turbo + VAD, true batched language detection

turbo_model = whisperx.load_model(
    'large-v3-turbo',
    device,
    compute_type=compute_type,
    download_root=model_turbo_dir,
)

vad_model, vad_utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    trust_repo=True,
    source='github',
)
vad_model = vad_model.to(device)
(get_speech_timestamps, _, _, _, _) = vad_utils

video_paths = []
for root, _, files in os.walk(SOURCE_ROOT):
    for file in files:
        if file.endswith('.mka'):
            video_paths.append(os.path.join(root, file))

for video_path in video_paths:
    bs = FILTER_INFER_BATCH_SIZE
    while True:
        try:
            filter_non_ro(video_path, infer_batch_size=bs)
            print(os.path.splitext(video_path)[0] + '.mask')
            break
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and bs > 8:
                bs = max(8, bs // 2)
                cuda_cleanup()
                print(f'OOM in filter for {video_path}, retrying with batch_size={bs}')
                continue
            raise

del turbo_model
del vad_model
cuda_cleanup()

# Align


In [ ]:
# Phase 2: load only large-v3, transcribe at batch size 32

model_big = whisperx.load_model(
    'large-v3',
    device,
    compute_type=compute_type,
    download_root=model_big_dir,
)

video_paths = []
for root, _, files in os.walk(SOURCE_ROOT):
    for file in files:
        if file.endswith('.mka'):
            video_paths.append(os.path.join(root, file))

for video_path in video_paths:
    transcribe(video_path, batch_size=TRANSCRIBE_BATCH_SIZE)
    print(os.path.splitext(video_path)[0] + '.json')

del model_big
cuda_cleanup()

In [ ]:
# Phase 3: load only align model

cuda_cleanup()

model_a, metadata = whisperx.load_align_model(
    language_code='ro',
    device=device,
    model_dir=model_a_dir,
)

In [ ]:
def align(path: str, batch_size: int = ALIGN_INFER_BATCH_SIZE) -> None:
    sr = 16000
    audio = whisperx.load_audio(path, sr=sr)

    mask_path = os.path.splitext(path)[0] + '.mask'
    mask = np.fromfile(mask_path, dtype=np.uint8)
    mask = np.unpackbits(mask)[: audio.shape[0]].astype(bool)
    audio[~mask] = 0.0

    result_path = os.path.splitext(path)[0] + '.json'
    with open(result_path, encoding='utf-8') as f:
        result = json.load(f)

    result = whisperx.align(
        result['segments'],
        model_a,
        metadata,
        audio,
        device,
        return_char_alignments=False,
        batch_size=batch_size,
    )

    out_path = os.path.splitext(path)[0] + '.align.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

In [ ]:
mask_paths = []
for root, _, files in os.walk(SOURCE_ROOT):
    for file in files:
        if file.endswith('.mask'):
            mask_paths.append(os.path.join(root, file))

for mask_path in mask_paths:
    align_path = mask_path.replace('.mask', '.align.json')

    if os.path.exists(align_path):
        print(f'skipping {mask_path}')
        continue

    video_path = mask_path.replace('.mask', '.mka')

    bs = ALIGN_INFER_BATCH_SIZE
    while True:
        try:
            align(video_path, batch_size=bs)
            print(mask_path)
            break
        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and bs > 8:
                bs = max(8, bs // 2)
                cuda_cleanup()
                print(f'OOM in align for {video_path}, retrying with batch_size={bs}')
                continue
            raise

del model_a
del metadata
cuda_cleanup()

## Filter hallucinations


In [ ]:
import json  # noqa: F811
import os
import re


def is_hallucination(text, min_repeats=3, dominance_threshold=0.5, max_phrase_len=64):
    tokens = re.findall(r'\b\w+\b', text.lower())
    total_tokens = len(tokens)

    for phrase_len in range(1, max_phrase_len + 1):
        i = 0
        while i <= total_tokens - phrase_len * min_repeats:
            # Get the current phrase
            phrase = tokens[i : i + phrase_len]
            count = 1

            # Check how many times it repeats consecutively
            while True:
                start = i + count * phrase_len
                end = start + phrase_len
                if end > total_tokens:
                    break
                if tokens[start:end] == phrase:
                    count += 1
                else:
                    break

            # If repeated enough times, check dominance
            if count >= min_repeats:
                total_phrase_tokens = count * phrase_len
                proportion = total_phrase_tokens / total_tokens
                if proportion >= dominance_threshold:
                    return True  # hallucination

                i += total_phrase_tokens  # skip ahead
            else:
                i += 1

    return False


def filter_garbage(segments):
    result = []

    for segment in segments:
        start = segment['start']
        end = segment['end']

        if len(segment['words']) == 0:
            continue

        duration = end - start
        text = (
            segment['text']
            .strip()
            .replace('ţ', 'ț')
            .replace('ş', 'ș')
            .replace('Ţ', 'Ț')
            .replace('Ş', 'Ș')
        )
        word_duration = duration / len(segment['words'])

        if duration < 2:
            continue

        if word_duration < 0.1:
            continue

        if word_duration > 2:
            continue

        if is_hallucination(text):
            continue

        result.append({'start': segment['start'], 'end': segment['end'], 'text': text})

    return result


for root, _, files in os.walk(SOURCE_ROOT):
    for file in files:
        if file.endswith('.align.json'):
            path = os.path.join(root, file)

            with open(path, encoding='utf-8') as f:
                js = json.load(f)
                segments = js['segments']

            segments = filter_garbage(segments)

            with open(
                path.replace('.align.json', '.align.tok.json'), 'w', encoding='utf-8'
            ) as f:
                json.dump(segments, f, ensure_ascii=False, indent=2)

            print(path.replace('.align.json', '.align.tok.json'))